# CNN on CIFAR-10

CIFAR-10 contains 60 000 colour images across 10 classes. This notebook trains a deeper CNN with batch normalisation and dropout, showing techniques needed when the task gets harder.

## Imports and data

We download CIFAR-10 — 60 000 colour images (32×32×3) across 10 classes (airplane, car, bird, etc.). We apply data augmentation (random flip and crop) to the training set and normalise both splits with per-channel mean and standard deviation to help the model converge faster.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = Path("../../data/raw")

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_ds = datasets.CIFAR10(data_dir, train=True, download=True, transform=transform_train)
test_ds = datasets.CIFAR10(data_dir, train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

class_names = train_ds.classes
print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Classes: {class_names}")

## Sample images

We display a few raw (unnormalised) training images to see the 32×32 colour pictures and their class labels.

In [ ]:
raw_ds = datasets.CIFAR10(data_dir, train=True, download=False, transform=transforms.ToTensor())
fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    img, label = raw_ds[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Define model

We build a deeper CNN with four convolutional layers, batch normalisation after each convolution, max-pooling to reduce spatial dimensions, and dropout for regularisation. Batch normalisation stabilises training, and dropout prevents the network from relying too heavily on any single neuron.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.MaxPool2d(2), nn.Dropout2d(0.25),

    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(2), nn.Dropout2d(0.25),

    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model

## Train and evaluate

We train for 10 epochs since CIFAR-10 is a harder task than MNIST. Each epoch runs the full training loop and measures test accuracy.

In [ ]:
history = {"epoch": [], "loss": [], "accuracy": []}

for epoch in range(1, 11):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = sum(
        (model(img.to(device)).argmax(1) == lab.to(device)).sum().item()
        for img, lab in test_loader
    )
    acc = correct / len(test_ds)
    avg_loss = total_loss / len(train_loader)

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["accuracy"].append(acc)
    print(f"Epoch {epoch:2d}: loss={avg_loss:.4f}, accuracy={acc:.3f}")

## Training curves

We plot loss and accuracy to check for convergence and potential overfitting. If the training loss keeps dropping but the test accuracy plateaus, more regularisation may be needed.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history["epoch"], history["loss"], marker="o")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss")
ax2.plot(history["epoch"], history["accuracy"], marker="o", color="green")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Test accuracy")
plt.tight_layout()
plt.show()

## Practical note

Colour images with higher variation need deeper networks and regularisation (batch norm, dropout, data augmentation). This small CNN reaches ~80-85 % on CIFAR-10; going further typically requires residual connections or pre-trained architectures.